## Creating Other Models for Benchmarking

### Parametric and Historical VaR

In [49]:
import os
import pandas as pd
import numpy as np
# (Assuming your other imports from earlier are still active)

# Re-introduce the export path from your very first snippet
path = r"C:\Users\nisha\Desktop\2026 PS-II\Projects\VaR\pythonProject1\data"
import_file = os.path.join(path, 'Results.csv')

sp500_data = pd.read_csv(import_file, index_col=0, parse_dates=True)

import scipy.stats as stats
import numpy as np

target_alpha = 0.01

# 1. Historical VaR
# Calculates the 1.5th percentile of actual returns over a rolling 252-day (1 trading year) window
sp500_data['Historical_VaR'] = sp500_data['Return'].rolling(window=252).quantile(target_alpha)

# 2. Parametric (Variance-Covariance) VaR
# Formula: VaR = Mean + (Z-Score * Volatility)
# We assume a daily mean of 0 (standard practice), and use your engineered RiskMetrics Volatility
z_score = stats.norm.ppf(target_alpha) # Gets the Z-score for 1.00% (approx -2.33)
sp500_data['Parametric_VaR'] = sp500_data['RiskMetrics_Vol'] * z_score

print(sp500_data[['Return', 'Historical_VaR', 'Parametric_VaR']])

              Return  Historical_VaR  Parametric_VaR
Date                                                
2001-05-16  2.805552             NaN       -3.105063
2001-05-17  0.272005             NaN       -3.408635
2001-05-18  0.268943             NaN       -3.308427
2001-05-21  1.602466             NaN       -3.211298
2001-05-22 -0.263133             NaN       -3.244614
...              ...             ...             ...
2026-04-15  0.794414       -2.173465       -2.614542
2026-04-16  0.260656       -2.173465       -2.574996
2026-04-17  1.196855       -2.173465       -2.500966
2026-04-20 -0.237720       -1.916812       -2.518865
2026-04-21 -0.636845       -1.916812       -2.445884

[6269 rows x 3 columns]


### Adding Polynomial and Linear Regression

In [50]:
import statsmodels.api as sm
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 1. Prepare 2D Regression Data 
reg_data = sp500_data[['Return', 'Rolling_GARCH_Vol', 'RiskMetrics_Vol', 'VIX_Daily_Shifted', 'RSI']].copy()
reg_data['Target_Return'] = reg_data['Return'].shift(-1)
reg_data.dropna(inplace=True)

# 2. Split exactly like the LSTM 
split_row = int(len(reg_data) * 0.7)
train_reg = reg_data.iloc[:split_row].copy()
test_reg  = reg_data.iloc[split_row:].copy()

features = ['Rolling_GARCH_Vol', 'RiskMetrics_Vol', 'VIX_Daily_Shifted', 'RSI']
X_tr = train_reg[features]
y_tr = train_reg['Target_Return']
X_te = test_reg[features]

# ===== NEW: STANDARDIZE FEATURES =====
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)
# =====================================

# 3. Linear Quantile Regression
X_tr_sm = sm.add_constant(X_tr_scaled)
X_te_sm = sm.add_constant(X_te_scaled)

# Fit with increased iterations
lin_model = sm.QuantReg(y_tr, X_tr_sm).fit(q=0.015, max_iter=2000)
test_reg.loc[:, 'QuantileReg_VaR'] = lin_model.predict(X_te_sm)

# 4. Polynomial Quantile Regression (Degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr_poly = poly.fit_transform(X_tr_scaled)  # ← Use scaled features
X_te_poly = poly.transform(X_te_scaled)

X_tr_poly_sm = sm.add_constant(X_tr_poly)
X_te_poly_sm = sm.add_constant(X_te_poly)

# Fit the polynomial model with increased iterations
poly_model = sm.QuantReg(y_tr, X_tr_poly_sm).fit(q=target_alpha, max_iter=2000)
test_reg.loc[:, 'PolyReg_VaR'] = poly_model.predict(X_te_poly_sm)

# Map the predictions back to the main dataframe
sp500_data['QuantileReg_VaR'] = np.nan  # ← Fixed typo from 'LinReg_VaR'
sp500_data['PolyReg_VaR'] = np.nan
sp500_data.loc[test_reg.index, 'QuantileReg_VaR'] = test_reg['QuantileReg_VaR']
sp500_data.loc[test_reg.index, 'PolyReg_VaR'] = test_reg['PolyReg_VaR']

print("Regression models fitted successfully.")

Regression models fitted successfully.


### Backtest

In [51]:
import pandas as pd
import numpy as np

# 1. Isolate the comparison window - keep only rows where ALL models have predictions
models = ['Historical_VaR', 'Parametric_VaR', 'QuantileReg_VaR', 'PolyReg_VaR', 'LSTM_VaR']
backtest_master = sp500_data[['Return'] + models].dropna().copy()

total_days = len(backtest_master)
results = []

for model_name in models:
    # Identify Breaches (Return is MORE NEGATIVE than VaR threshold)
    series = (backtest_master['Return'] < backtest_master[model_name]).astype(int)
    breaches = series.sum()
    failure_rate = breaches / total_days
    
    results.append({
        'Model': model_name,
        'Actual Breaches': int(breaches),  # ← Ensure it's an int
        'Failure Rate (%)': round(failure_rate * 100, 2),
    })

# 3. Display the Enhanced Report
results_df = pd.DataFrame(results).sort_values(by='Failure Rate (%)')
print(f"--- BACKTEST WITH CLUSTERING ANALYSIS ---")
print(f"Total Days: {total_days} | Target Alpha: {target_alpha*100}%\n")
print(results_df.to_string(index=False))

export_file = os.path.join(path, 'Final_Results.csv')
sp500_data.to_csv(export_file)

--- BACKTEST WITH CLUSTERING ANALYSIS ---
Total Days: 1881 | Target Alpha: 1.0%

          Model  Actual Breaches  Failure Rate (%)
       LSTM_VaR               17              0.90
QuantileReg_VaR               26              1.38
    PolyReg_VaR               28              1.49
 Historical_VaR               29              1.54
 Parametric_VaR               46              2.45
